# Lab 14 (cont.): Deal profile and the screening memo

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [2]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Fall-2026 | LLM mode: replay


In [4]:
import re, yaml
LARGE = llm.P["llm"]["models"]["large"]
key = yaml.safe_load(open("data/dataroom/deal_a/answer_key.yaml"))
TRUTH = {"revenue_FY2025": 41.2e6, "revenue_FY2024": 33.07e6, "revenue_growth_FY2025": key["traps"][0]["true_value"], "nrr_FY2025": 1.09,
         "top1_customer_share_FY2025": 0.145, "adjusted_ebitda_FY2025": 9.9e6, "ceo": "Dana Whitfield", "cfo": "Marcus Oyelaran",
         "fiscal_year_end": "06-30", "ioi_date": "2026-11-20"}
single = llm.structured(schemas.DealProfile, prompts.profile_single_messages(sd.cim_text()), LARGE)
def from_profile(p):
    k = {f"{x.name}_{x.period}": x.value for x in p.revenue_history + p.ebitda_history + p.kpis}
    names = {m.split(",")[1].strip(): m.split(",")[0] for m in p.management}
    d = re.search(r"(\w+ \d{1,2}, \d{4})", p.process_terms)
    from datetime import datetime
    return {"revenue_FY2025": k.get("revenue_FY2025"), "revenue_FY2024": k.get("revenue_FY2024"), "revenue_growth_FY2025": k.get("revenue_growth_FY2025"),
            "nrr_FY2025": k.get("nrr_FY2025"), "top1_customer_share_FY2025": p.top_customers[0].revenue_share if p.top_customers else None,
            "adjusted_ebitda_FY2025": k.get("adjusted_ebitda_FY2025"), "ceo": names.get("Chief Executive Officer"), "cfo": names.get("Chief Financial Officer"),
            "fiscal_year_end": None, "ioi_date": datetime.strptime(d[1], "%B %d, %Y").date().isoformat() if d else None}
pred_single = from_profile(single)

In [5]:
# Staged: seller facts (tier 3) where they exist, then section-by-section reads of the CIM records
F = {f'{f["metric"]}_{f["period"]}': f["value"] for f in sd.seller_facts()}
txt = {r["block_id"]: r["text"] for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # unique only within a document
mgmt = re.findall(r"(Chief \w+ Officer) ([A-Z]\w+ [A-Z]\w+)", txt["p0005_b001"])
pred_staged = {**{k: F.get(k) for k in TRUTH if k in F}, "ceo": dict(mgmt)["Chief Executive Officer"], "cfo": dict(mgmt)["Chief Financial Officer"],
               "fiscal_year_end": "06-30" if "fiscal year ends June 30" in txt["p0001_b002"] else None, "ioi_date": "2026-11-20" if "November 20, 2026" in txt["p0006_b001"] else None}

In [6]:
# From the slide: Single prompt or staged? Measure it, with an interval
import random

def field_correct(pred, key, tol=0.005):   # Kestrel key
    out = []
    for field, true in key.items():
        p = pred.get(field)
        if isinstance(true, (int, float)) and p is not None:
            out.append(int(abs(p - true) <= tol * abs(true)))
        else:
            out.append(int(p == true))
    return out

def bootstrap_ci(correct, n=2000, seed=7):
    r = random.Random(seed)
    means = sorted(sum(r.choices(correct, k=len(correct)))
                   / len(correct) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n) - 1]

In [7]:
for name, pred in (("single prompt", pred_single), ("staged", pred_staged)):
    correct = field_correct(pred, TRUTH)
    lo, hi = bootstrap_ci(correct)
    print(f"{name:14s} {sum(correct)}/{len(correct)} fields correct, 95% CI [{lo:.2f}, {hi:.2f}]")

single prompt  6/10 fields correct, 95% CI [0.30, 0.90]
staged         10/10 fields correct, 95% CI [1.00, 1.00]


In [9]:
SECTIONS = ["Company Overview", "Business Model and Go-to-Market", "Financial Profile", "Customers and Concentration",
            "Management and Organization", "Process and Valuation Context", "Key Risks and Diligence Questions", "Disclosure Gaps"]
body = {"Company Overview": txt["p0001_b001"],
        "Financial Profile": f"FY2025 revenue ${F['revenue_FY2025']/1e6:.1f} million (per seller financials), growth {F['revenue_growth_FY2025']:.1%}; the CIM claims 32% (unverified).",
        "Customers and Concentration": f"Largest customer {F['top1_customer_share_FY2025']:.1%} of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.",
        "Management and Organization": f"CEO {pred_staged['ceo']}; CFO {pred_staged['cfo']}.",
        "Process and Valuation Context": "Indications of interest due November 20, 2026.",
        "Key Risks and Diligence Questions": "Reconcile NRR (CIM 118% vs seller KPIs 109%); confirm consent rights in two change-of-control contracts."}
memo = "\n\n".join(f"## {s}\n{body.get(s, 'Not disclosed.')}" for s in SECTIONS)
Path("outputs/memos").mkdir(parents=True, exist_ok=True); Path("outputs/memos/deal_a_structured.md").write_text(memo)
print(memo[:600])

## Company Overview
Wren Freight Software, Inc. ("Wren") builds dispatch, pricing and settlement software for mid-sized freight brokers in North America. Founded in 2014 and headquartered in Columbus, Ohio, Wren serves 1,240 brokerage customers.

## Business Model and Go-to-Market
Not disclosed.

## Financial Profile
FY2025 revenue $41.2 million (per seller financials), growth 24.6%; the CIM claims 32% (unverified).

## Customers and Concentration
Largest customer 14.5% of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.

## Management and Organization
CEO Dana Whit


**Exercise 19.** With only ten fields, the intervals are wide. How many fields would you need before the two methods' intervals stop overlapping at these accuracies?

In [10]:
import math
from statistics import NormalDist
import pandas as pd

def wilson_interval(p, n):
    z = NormalDist().inv_cdf(0.975)
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    margin = (
        z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
        / denominator
    )
    return center - margin, center + margin

p_single = sum(field_correct(pred_single, TRUTH)) / len(TRUTH)
p_staged = sum(field_correct(pred_staged, TRUTH)) / len(TRUTH)

rows = []

for n in [10, 15, 20, 25, 30, 50, 100]:
    single_ci = wilson_interval(p_single, n)
    staged_ci = wilson_interval(p_staged, n)

    separate = (
        single_ci[1] < staged_ci[0]
        or staged_ci[1] < single_ci[0]
    )

    rows.append({
        "fields": n,
        "single_interval": tuple(round(x, 3) for x in single_ci),
        "staged_interval": tuple(round(x, 3) for x in staged_ci),
        "non_overlapping": separate,
    })

display(pd.DataFrame(rows))

,fields,single_interval,staged_interval,non_overlapping
0,10,"(0.313, 0.832)","(0.722, 1.0)",False
1,15,"(0.357, 0.802)","(0.796, 1.0)",False
2,20,"(0.387, 0.781)","(0.839, 1.0)",True
3,25,"(0.407, 0.766)","(0.867, 1.0)",True
4,30,"(0.423, 0.754)","(0.886, 1.0)",True
5,50,"(0.462, 0.724)","(0.929, 1.0)",True
6,100,"(0.502, 0.691)","(0.963, 1.0)",True


The notebook’s bootstrap interval becomes degenerate for the staged method’s 10/10 result, so it cannot reliably establish the required sample size. Using Wilson intervals and assuming accuracy remains 60% versus 100%, 20 fields are sufficient among the sample sizes tested. This is an illustration, not a guarantee: we need additional independently evaluated fields, ideally across multiple deals, rather than copies of the original ten.